# Budgerigar：听取—思考—复读行为评估

本 notebook 不继续训练。它检查网络是否真的学会在听取和思考期间保持静默，并在复读区间发声，而不是仅依靠聚合 loss。

In [ ]:
#@title 1. 更新项目与安装依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并定位训练产物
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
TARGET_SPEAKER='arctic_slt' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
RUN_DIR=WORK_ROOT/'checkpoints'/f'neural_echo_{TARGET_SPEAKER}_{FEATURE_FINGERPRINT}'
CHECKPOINT=RUN_DIR/'best.pt'
assert FEATURE_MANIFEST.is_file(),FEATURE_MANIFEST
assert CHECKPOINT.is_file(),CHECKPOINT
print(CHECKPOINT)

In [ ]:
#@title 3. 运行验证集时间轴评估
MAX_PAIRS=64 #@param {type:'integer'}
VOICE_THRESHOLD=0.5 #@param {type:'number'}
import json,torch
from budgerigar.evaluate_echo import evaluate_checkpoint
EVAL_DIR=RUN_DIR/'behavior_evaluation'
report=evaluate_checkpoint(CHECKPOINT,FEATURE_MANIFEST,EVAL_DIR,max_pairs=MAX_PAIRS,threshold=VOICE_THRESHOLD)
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 绘制神经发声强度时间轴
import matplotlib.pyplot as plt
examples=torch.load(EVAL_DIR/'behavior_examples.pt',map_location='cpu',weights_only=True)
fig,axes=plt.subplots(min(4,len(examples)),1,figsize=(15,3*min(4,len(examples))),squeeze=False)
for axis,example in zip(axes[:,0],examples[:4]):
    meta=example['metadata']; probability=example['probability']; target=example['target_voice']
    axis.plot(probability.numpy(),label='predicted voice probability')
    axis.plot(target.numpy(),alpha=.65,label='target voice')
    axis.axvline(meta['source_frames'],color='orange',linestyle='--',label='source signal ends')
    axis.axvspan(meta['source_frames'],meta['repeat_start'],color='gold',alpha=.18,label='thinking interval')
    axis.axvline(meta['repeat_start'],color='green',linestyle='--',label='target repeat begins')
    axis.set_title(meta['source_id']);axis.set_ylim(-.05,1.05);axis.legend(loc='upper right')
plt.tight_layout();plt.show()

In [ ]:
#@title 5. 阶段判定
print('behavior_pass =',report['behavior_pass'])
if report['silent_output_rate']>=0.1: print('失败模式：模型可能退化为全程静默')
if report['false_voice_rate']['listen']>=0.05: print('失败模式：模型在输入尚未完整时提前发声')
if report['false_voice_rate']['thinking']>=0.05: print('失败模式：模型没有保留思考时间')
if report['repeat_voiced_recall']<=0.5: print('失败模式：复读区间的发声召回不足')
print('报告：',EVAL_DIR/'behavior_report.json')

In [ ]:
#@title 6. 保存评估运行元数据
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(EVAL_DIR/'run_metadata.json',FEATURE_MANIFEST,{'evaluation':'neural_echo_timeline','checkpoint':str(CHECKPOINT),'behavior_pass':report['behavior_pass']},repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))